In [28]:
import pandas as pd
import sqlite3

<h2>Создайте подключение к базе данных с помощью sqlite3 библиотеки.</h2>

In [29]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

<h2>Используя только один запрос для каждой группы, создайте два фрейма данных: test_results и control_results, со столбцами «time» и «avg_diff» и только двумя строками.</h2>

- «время» должно иметь значения после и до.
- «avg_diff» должен содержать среднюю дельту для всех пользователей за период времени до и после их первого посещения страницы.
- Учитывать только пользователей, имевших наблюдения до и после.


- Мы все еще не пользуемся лабораторией project1

In [30]:
query_test = """
    SELECT time, AVG(avg_diff) AS avg_diff
    FROM (
        SELECT t.uid,
               CASE
                   WHEN t.first_commit_ts < t.first_view_ts THEN 'before'
                   ELSE 'after'
               END AS time,
               AVG((CAST(strftime('%s', t.first_commit_ts) AS INT) - d.deadlines) / 3600) AS avg_diff
        FROM test t
        JOIN deadlines d ON t.labname = d.labs
        WHERE t.labname != 'project1'
        GROUP BY t.uid, time
    ) sub
    WHERE uid IN (
        SELECT uid
        FROM (
            SELECT t.uid, COUNT(DISTINCT CASE WHEN t.first_commit_ts < t.first_view_ts THEN 'before' ELSE 'after' END) as cnt
            FROM test t
            JOIN deadlines d ON t.labname = d.labs
            WHERE t.labname != 'project1'
            GROUP BY t.uid
        ) q
        WHERE cnt = 2
    )
    GROUP BY time;
"""

In [31]:
query_control = """
    SELECT time, AVG(avg_diff) AS avg_diff
    FROM (
        SELECT c.uid,
               CASE
                   WHEN c.first_commit_ts < c.first_view_ts THEN 'before'
                   ELSE 'after'
               END AS time,
               AVG((CAST(strftime('%s', c.first_commit_ts) AS INT) - d.deadlines) / 3600) AS avg_diff
        FROM control c
        JOIN deadlines d ON c.labname = d.labs
        WHERE c.labname != 'project1'
        GROUP BY c.uid, time
    ) sub
    WHERE uid IN (
        SELECT uid
        FROM (
            SELECT c.uid, COUNT(DISTINCT CASE WHEN c.first_commit_ts < c.first_view_ts THEN 'before' ELSE 'after' END) as cnt
            FROM control c
            JOIN deadlines d ON c.labname = d.labs
            WHERE c.labname != 'project1'
            GROUP BY c.uid
        ) q
        WHERE cnt = 2
    )
    GROUP BY time;
"""

In [32]:
test_results = pd.read_sql(query_test, conn)

In [33]:
control_results = pd.read_sql(query_control, conn)

In [34]:
test_results.head()

,time,avg_diff
0,after,-99.523810
1,before,-66.047619


In [35]:
control_results.head()

,time,avg_diff
0,after,-99.322222
1,before,-98.033333


<h2>Закройте соединение.</h2>

In [36]:
conn.close()

<h2> Получите ответ: подтвердилась ли гипотеза, и влияет ли страница на поведение студентов? </h2>

Гипотеза подтвердилась:

- В test-группе дельта заметно выросла после первого визита в новостную ленту. (~33 часа)

- В control-группе роста практически нет(~1 час)